In [1]:
import sys

print(sys.executable)

/mnt/d809bc61-ebac-4ca7-8e1e-e0a6d7e8f201/Documents/Programming/MyProjects/transformer-news-classification/.venv/bin/python


In [5]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_ag_news

dataset = load_ag_news()

dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [6]:
from collections import Counter

train_labels = dataset["train"]["label"]
test_labels = dataset["test"]["label"]

print("Train:", Counter(train_labels))
print("Test:", Counter(test_labels))

Train: Counter({2: 30000, 3: 30000, 1: 30000, 0: 30000})
Test: Counter({2: 1900, 3: 1900, 1: 1900, 0: 1900})


In [7]:
print(dataset["train"].features)

{'text': Value('string'), 'label': ClassLabel(names=['World', 'Sports', 'Business', 'Sci/Tech'])}


In [8]:
label_names = dataset["train"].features["label"].names

for label in range(4):
    print("=" * 80)
    print(f"CLASS {label}: {label_names[label]}")
    print("=" * 80)

    sample = dataset["train"].filter(
        lambda x: x["label"] == label
    ).select(range(3))

    for i, text in enumerate(sample["text"], start=1):
        print(f"\nExample {i}:")
        print(text)

CLASS 0: World


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]


Example 1:
Venezuelans Vote Early in Referendum on Chavez Rule (Reuters) Reuters - Venezuelans turned out early\and in large numbers on Sunday to vote in a historic referendum\that will either remove left-wing President Hugo Chavez from\office or give him a new mandate to govern for the next two\years.

Example 2:
S.Koreans Clash with Police on Iraq Troop Dispatch (Reuters) Reuters - South Korean police used water cannon in\central Seoul Sunday to disperse at least 7,000 protesters\urging the government to reverse a controversial decision to\send more troops to Iraq.

Example 3:
Palestinians in Israeli Jails Start Hunger Strike (Reuters) Reuters - Thousands of Palestinian\prisoners in Israeli jails began a hunger strike for better\conditions Sunday, but Israel's security minister said he\didn't care if they starved to death.
CLASS 1: Sports


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]


Example 1:
Phelps, Thorpe Advance in 200 Freestyle (AP) AP - Michael Phelps took care of qualifying for the Olympic 200-meter freestyle semifinals Sunday, and then found out he had been added to the American team for the evening's 400 freestyle relay final. Phelps' rivals Ian Thorpe and Pieter van den Hoogenband and teammate Klete Keller were faster than the teenager in the 200 free preliminaries.

Example 2:
Reds Knock Padres Out of Wild-Card Lead (AP) AP - Wily Mo Pena homered twice and drove in four runs, helping the Cincinnati Reds beat the San Diego Padres 11-5 on Saturday night. San Diego was knocked out of a share of the NL wild-card lead with the loss and Chicago's victory over Los Angeles earlier in the day.

Example 3:
Dreaming done, NBA stars awaken to harsh Olympic reality (AFP) AFP - National Basketball Association players trying to win a fourth consecutive Olympic gold medal for the United States have gotten the wake-up call that the "Dream Team" days are done even if su

Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]


Example 1:
Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.

Example 2:
Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.

Example 3:
Oil and Economy Cloud Stocks' Outlook (Reuters) Reuters - Soaring crude prices plus worries\about the economy and the outlook for earnings are expected to\hang over the stock market next week during the depth of the\summer doldrums.
CLASS 3: Sci/Tech


Filter:   0%|          | 0/120000 [00:00<?, ? examples/s]


Example 1:
'Madden,' 'ESPN' Football Score in Different Ways (Reuters) Reuters - Was absenteeism a little high\on Tuesday among the guys at the office? EA Sports would like\to think it was because "Madden NFL 2005" came out that day,\and some fans of the football simulation are rabid enough to\take a sick day to play it.

Example 2:
Group to Propose New High-Speed Wireless Format (Reuters) Reuters - A group of technology companies\including Texas Instruments Inc. (TXN.N), STMicroelectronics\(STM.PA) and Broadcom Corp. (BRCM.O), on Thursday said they\will propose a new wireless networking standard up to 10 times\the speed of the current generation.

Example 3:
AOL to Sell Cheap PCs to Minorities and Seniors (Reuters) Reuters - America Online on Thursday said it\plans to sell a low-priced PC targeting low-income and minority\households who agree to sign up for a year of dialup Internet\service.


In [9]:
train_texts = dataset["train"]["text"]

empty_count = sum(
    1 for text in train_texts
    if text is None or not text.strip()
)

print("Total training samples:", len(train_texts))
print("Empty or None texts:", empty_count)

Total training samples: 120000
Empty or None texts: 0


In [10]:
unique_texts = len(set(train_texts))
duplicate_count = len(train_texts) - unique_texts

print("Total training samples:", len(train_texts))
print("Unique texts:", unique_texts)
print("Duplicate texts:", duplicate_count)

Total training samples: 120000
Unique texts: 120000
Duplicate texts: 0


In [11]:
text_lengths = [len(text) for text in train_texts]

print("Minimum characters:", min(text_lengths))
print("Maximum characters:", max(text_lengths))
print("Average characters:", sum(text_lengths) / len(text_lengths))

Minimum characters: 100
Maximum characters: 1012
Average characters: 236.477525


In [12]:
import numpy as np

percentiles = np.percentile(
    text_lengths,
    [50, 90, 95, 99]
)

print("50th percentile:", percentiles[0])
print("90th percentile:", percentiles[1])
print("95th percentile:", percentiles[2])
print("99th percentile:", percentiles[3])

50th percentile: 232.0
90th percentile: 300.0
95th percentile: 343.0
99th percentile: 464.0


In [13]:
for label in range(4):
    class_lengths = [
        len(text)
        for text, current_label in zip(
            train_texts,
            dataset["train"]["label"]
        )
        if current_label == label
    ]

    print(
        label_names[label],
        "-> Average characters:",
        sum(class_lengths) / len(class_lengths)
    )

World -> Average characters: 242.62666666666667
Sports -> Average characters: 224.6522
Business -> Average characters: 241.40856666666667
Sci/Tech -> Average characters: 237.22266666666667
